# Go to very last u will get the best way of hyper parameter tuning

### Without any hyperparameter tuning - finding accuracy using cross validation

In [2]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier

# Load Phoneme dataset (5,404 rows, 5 purely numeric features, 0 missing values)
df = fetch_openml(name='phoneme', version=1, as_frame=True).frame

X = df.drop(columns=['Class'])
y = df['Class']  # Output column: '1' (nasal) vs '2' (oral)


In [3]:
df.head()

,V1,V2,V3,V4,V5,Class
0,0.489927,-0.451528,-1.047990,-0.598693,-0.020418,1
1,-0.641265,0.109245,0.292130,-0.916804,0.240223,1
2,0.870593,-0.459862,0.578159,0.806634,0.835248,1
3,-0.628439,-0.316284,1.934295,-1.427099,-0.136583,1
4,-0.596399,0.015938,2.043206,-1.688448,-0.948127,1


In [4]:
df['Class'].value_counts()

,count
Class,
1,3818
2,1586


In [5]:
model = KNeighborsClassifier()
stf = StratifiedKFold(
    n_splits = 5,
    shuffle = True,
    random_state = 42
)

scores = cross_val_score(
    model,
    X,
    y,
    cv = stf,
    scoring = 'accuracy'
)

scores.mean()

np.float64(0.8771300579024908)

# 1. GridSearchCV

In [6]:
from sklearn.model_selection import GridSearchCV

model = KNeighborsClassifier()
stf = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42 )

param_grid = {
    'n_neighbors':[1,3,5,7,10,12,15,17,20],
    'weights':['uniform','distance'],
    'algorithm':['ball_tree', 'kd_tree', 'brute'],
    'p':[1,2]
}

gcv = GridSearchCV(model, param_grid, scoring='accuracy', refit=True, cv=stf, verbose=2)

In [7]:
gcv.fit(X,y)

Fitting 5 folds for each of 108 candidates, totalling 540 fits
[CV] END algorithm=ball_tree, n_neighbors=1, p=1, weights=uniform; total time=   0.2s
[CV] END algorithm=ball_tree, n_neighbors=1, p=1, weights=uniform; total time=   0.1s
[CV] END algorithm=ball_tree, n_neighbors=1, p=1, weights=uniform; total time=   0.1s
[CV] END algorithm=ball_tree, n_neighbors=1, p=1, weights=uniform; total time=   0.2s
[CV] END algorithm=ball_tree, n_neighbors=1, p=1, weights=uniform; total time=   0.2s
[CV] END algorithm=ball_tree, n_neighbors=1, p=1, weights=distance; total time=   0.3s
[CV] END algorithm=ball_tree, n_neighbors=1, p=1, weights=distance; total time=   0.1s
[CV] END algorithm=ball_tree, n_neighbors=1, p=1, weights=distance; total time=   0.1s
[CV] END algorithm=ball_tree, n_neighbors=1, p=1, weights=distance; total time=   0.1s
[CV] END algorithm=ball_tree, n_neighbors=1, p=1, weights=distance; total time=   0.1s
[CV] END algorithm=ball_tree, n_neighbors=1, p=2, weights=uniform; total

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=KNeighborsClassifier(),
             param_grid={'algorithm': ['ball_tree', 'kd_tree', 'brute'],
                         'n_neighbors': [1, 3, 5, 7, 10, 12, 15, 17, 20],
                         'p': [1, 2], 'weights': ['uniform', 'distance']},
             scoring='accuracy', verbose=2)

In [8]:
# finding best parameters for the model
gcv.best_params_

{'algorithm': 'ball_tree', 'n_neighbors': 3, 'p': 1, 'weights': 'distance'}

In [9]:
# finding best score - the above parameter gives the best score
gcv.best_score_

np.float64(0.8995187926131496)

In [10]:
gcv.cv_results_

# this gives the overall results , u can see this when u convert it into dataframe

{'mean_fit_time': array([0.03568006, 0.02801504, 0.01864972, 0.01360893, 0.02175159,
        0.01744976, 0.01852708, 0.02674499, 0.02273636, 0.01997585,
        0.02276497, 0.0219305 , 0.02268124, 0.02472839, 0.02240829,
        0.00924764, 0.0285543 , 0.02749119, 0.03649826, 0.03975801,
        0.01398134, 0.01387486, 0.01403546, 0.01411581, 0.01023059,
        0.00841265, 0.00924058, 0.00861511, 0.00847063, 0.00822153,
        0.00840082, 0.01011515, 0.0091733 , 0.00930614, 0.00865774,
        0.01033144, 0.00891871, 0.00902205, 0.00911889, 0.00925293,
        0.00892634, 0.00962515, 0.00884342, 0.01020622, 0.00962243,
        0.00912495, 0.00932856, 0.00907111, 0.01010833, 0.00888548,
        0.00862832, 0.01025286, 0.00942883, 0.00986071, 0.00919895,
        0.00976844, 0.00958223, 0.00930634, 0.0095458 , 0.00944877,
        0.00870991, 0.00981736, 0.00969071, 0.00999274, 0.00876517,
        0.0089426 , 0.00898256, 0.00949373, 0.0090795 , 0.0110023 ,
        0.00989208, 0.00871968,

In [11]:
import pandas as pd
pd.DataFrame(gcv.cv_results_)[['param_algorithm',	'param_n_neighbors',	'param_p', 'param_weights', 'mean_test_score']].sort_values('mean_test_score',ascending=False)

,param_algorithm,param_n_neighbors,param_p,param_weights,mean_test_score
5,ball_tree,3,1,distance,0.899519
41,kd_tree,3,1,distance,0.899519
77,brute,3,1,distance,0.899519
2,ball_tree,1,2,uniform,0.898227
39,kd_tree,1,2,distance,0.898227
...,...,...,...,...,...
30,ball_tree,17,2,uniform,0.854184
66,kd_tree,17,2,uniform,0.854184
34,ball_tree,20,2,uniform,0.848261
70,kd_tree,20,2,uniform,0.848261


In [12]:
# this tables gives us an idea that if we a almost 2 similar accuracies but there cost is high and low , then we can select the low one
# eg - if n_neigher = 12 has accuracy of 0.89 and n_neighbor = 3 has accuracy 0.88 then we can choose the 2nd on ebcz its cost is less only 3 neighbour calculation

In [13]:
gcv.best_index_

# it gives use the best index from the above table(cv_results_)

np.int64(5)

In [17]:
gcv.best_estimator_

KNeighborsClassifier(algorithm='ball_tree', n_neighbors=3, p=1,
                     weights='distance')

# 2. RandomizedSearchCV

In [14]:
from sklearn.model_selection import RandomizedSearchCV

model = KNeighborsClassifier()
stf = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42 )

param_grid = {
    'n_neighbors':[1,3,5,7,10,12,15,17,20],
    'weights':['uniform','distance'],
    'algorithm':['ball_tree', 'kd_tree', 'brute'],
    'p':[1,2]
}

rcv = RandomizedSearchCV(model, param_grid, scoring='accuracy', refit=True, cv=stf, verbose=2 , n_iter = 15)

In [15]:
rcv.fit(X,y)

Fitting 5 folds for each of 15 candidates, totalling 75 fits
[CV] END algorithm=kd_tree, n_neighbors=20, p=2, weights=uniform; total time=   0.0s
[CV] END algorithm=kd_tree, n_neighbors=20, p=2, weights=uniform; total time=   0.0s
[CV] END algorithm=kd_tree, n_neighbors=20, p=2, weights=uniform; total time=   0.0s
[CV] END algorithm=kd_tree, n_neighbors=20, p=2, weights=uniform; total time=   0.0s
[CV] END algorithm=kd_tree, n_neighbors=20, p=2, weights=uniform; total time=   0.0s
[CV] END algorithm=brute, n_neighbors=7, p=1, weights=uniform; total time=   0.1s
[CV] END algorithm=brute, n_neighbors=7, p=1, weights=uniform; total time=   0.1s
[CV] END algorithm=brute, n_neighbors=7, p=1, weights=uniform; total time=   0.1s
[CV] END algorithm=brute, n_neighbors=7, p=1, weights=uniform; total time=   0.1s
[CV] END algorithm=brute, n_neighbors=7, p=1, weights=uniform; total time=   0.1s
[CV] END algorithm=brute, n_neighbors=1, p=2, weights=uniform; total time=   0.0s
[CV] END algorithm=bru

RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=KNeighborsClassifier(), n_iter=15,
                   param_distributions={'algorithm': ['ball_tree', 'kd_tree',
                                                      'brute'],
                                        'n_neighbors': [1, 3, 5, 7, 10, 12, 15,
                                                        17, 20],
                                        'p': [1, 2],
                                        'weights': ['uniform', 'distance']},
                   scoring='accuracy', verbose=2)

In [18]:
rcv.best_params_

{'weights': 'uniform', 'p': 2, 'n_neighbors': 1, 'algorithm': 'brute'}

In [19]:
rcv.best_score_

np.float64(0.8982267790454653)

In [20]:
rcv.cv_results_

{'mean_fit_time': array([0.00945992, 0.00622005, 0.00579696, 0.00583344, 0.00944819,
        0.00735955, 0.00847559, 0.00820169, 0.00616474, 0.00901904,
        0.00646982, 0.00870442, 0.00831022, 0.00896659, 0.00882411]),
 'std_fit_time': array([0.00139083, 0.00044305, 0.00038929, 0.00029707, 0.00072181,
        0.00252712, 0.00022803, 0.0002164 , 0.00070876, 0.00028609,
        0.00155873, 0.00022018, 0.00064217, 0.00060841, 0.00162392]),
 'mean_score_time': array([0.02292109, 0.07199488, 0.02917728, 0.07090807, 0.01162901,
        0.03289099, 0.02260385, 0.02772322, 0.07062149, 0.01284099,
        0.07056708, 0.02850437, 0.0307394 , 0.01760755, 0.10091124]),
 'std_score_time': array([0.00064479, 0.00489584, 0.00073935, 0.00089813, 0.00194551,
        0.00248146, 0.00092106, 0.0012583 , 0.00088532, 0.00051172,
        0.00225614, 0.00210964, 0.00063003, 0.00053929, 0.01219089]),
 'param_weights': masked_array(data=['uniform', 'uniform', 'uniform', 'uniform', 'uniform',
              

In [23]:
pd.DataFrame(rcv.cv_results_)[['param_algorithm',	'param_n_neighbors',	'param_p', 'param_weights', 'mean_test_score']].sort_values('mean_test_score',ascending=False)

,param_algorithm,param_n_neighbors,param_p,param_weights,mean_test_score
2,brute,1,2,uniform,0.898227
6,ball_tree,1,2,distance,0.898227
4,kd_tree,1,2,uniform,0.898227
9,kd_tree,3,2,distance,0.898040
12,ball_tree,10,2,distance,0.896189
14,brute,15,1,distance,0.895819
7,ball_tree,3,1,uniform,0.890636
5,brute,5,2,uniform,0.877130
1,brute,7,1,uniform,0.874355
8,brute,15,1,uniform,0.862327


# 3. Bayesian Search (Optuna)

- It is done by the optuna framework
- It is the most important part , we are going to use it everywhere

## Steps of Optuna
- 1. Import: import optuna along with your ML library.
- 2. Objective: Define **objective(trial)** function to suggest hyperparameters via trial.suggest_*() **bold text** and return an evaluation score.
- 3. Study: Create a study with study = **optuna.create_study(direction="maximize") (or "minimize")**.
- 4. Optimize: Run trials using **study.optimize(objective, n_trials=...)**.
- 5. Results: Retrieve the optimal configuration via **study.best_params** and **study.best_value.**

In [26]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [optuna]


In [27]:
from sklearn.datasets import load_breast_cancer

df = load_breast_cancer(as_frame=True).frame
df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [29]:
import optuna
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd


# Split into features (X) and target (y)
X = df.drop(columns=['target'])
y = df['target']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')

Training set shape: (398, 30)
Test set shape: (171, 30)


In [ ]:
# Step 2 - Make the objective function

In [51]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import optuna

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimator' , 50, 200)  # integers
    max_depth = trial.suggest_int('max_depth' , 3 , 20)        # integers
    max_features = trial.suggest_float("max_features", 0.1, 1.0)   # float
    criterion = trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"]) # categorical


    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth = max_depth,
        max_features = max_features,
        criterion = criterion ,
        random_state=42
    )

    # Perform any cross-validation and calculate accuracy
    score = cross_val_score(model , X , y , scoring = 'accuracy' , cv = 5).mean()

    return score # Return the accuracy score for Optuna to maximize


In [ ]:
# step 3 - Study: Create a study with study = optuna.create_study(direction="maximize") (or "minimize").

In [52]:
# Create a study object and optimize the objective function

study = optuna.create_study(
    direction = 'maximize',      # bcz we want maximum accuracy , if scoring was MSE then we use minimize
    sampler = optuna.samplers.TPESampler()    # TPE sampler use baysian search ( we can use RandomSampler or GridSampler for random seach or grid search)
)

[I 2026-11-24 11:24:27,577] A new study created in memory with name: no-name-b10dc0fe-47e9-4320-9c54-e32d6dc5556e


In [ ]:
# step 4 - Optimize: Run trials using study.optimize(objective, n_trials=...).

In [53]:
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2026-11-24 11:24:38,833] Trial 0 finished with value: 0.9560782487191428 and parameters: {'n_estimator': 55, 'max_depth': 14, 'max_features': 0.7009444041850462, 'criterion': 'log_loss'}. Best is trial 0 with value: 0.9560782487191428.
[I 2026-11-24 11:24:41,575] Trial 1 finished with value: 0.9683744760130415 and parameters: {'n_estimator': 121, 'max_depth': 18, 'max_features': 0.4323397274618712, 'criterion': 'log_loss'}. Best is trial 1 with value: 0.9683744760130415.
[I 2026-11-24 11:24:43,396] Trial 2 finished with value: 0.9526005278683434 and parameters: {'n_estimator': 50, 'max_depth': 7, 'max_features': 0.8036756864142842, 'criterion': 'gini'}. Best is trial 1 with value: 0.9683744760130415.
[I 2026-11-24 11:24:49,396] Trial 3 finished with value: 0.9561092997981678 and parameters: {'n_estimator': 148, 'max_depth': 9, 'max_features': 0.8078743703610226, 'criterion': 'gini'}. Best is trial 1 with value: 0.9683744760130415.
[I 2026-11-24 11:24:50,938] Trial 4 finished with va

In [35]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.9701443875174661
Best hyperparameters: {'n_estimator': 160, 'max_depth': 17, 'max_features': 0.7546815784556188, 'criterion': 'log_loss'}


In [42]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(n_estimators= 160, max_depth= 17, max_features = 0.7546815784556188, criterion = 'log_loss' , random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.96


In [ ]:
# without any hyper parameter , we were at 87% accuracy , after hyperparameter tuning we are at 96

### Using random sampler in optuna

In [43]:
# Uncomment all to run this


# import optuna
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.model_selection import cross_val_score


# # Define the objective function
# def objective(trial):
#     n_estimators = trial.suggest_int("n_estimators", 50, 200)
#     max_depth = trial.suggest_int("max_depth", 3, 20)
#     max_features = trial.suggest_float("max_features", 0.1, 1.0)
#     criterion = trial.suggest_categorical(
#         "criterion", ["gini", "entropy", "log_loss"]
#     )

#     model = RandomForestClassifier(
#         n_estimators=n_estimators,
#         max_depth=max_depth,
#         max_features=max_features,
#         criterion=criterion,
#         random_state=42,
#     )

#     score = cross_val_score(model, X, y, scoring="accuracy", cv=5).mean()
#     return score


# # Create study with RandomSampler
# study_random = optuna.create_study(
#     direction="maximize", sampler=optuna.samplers.RandomSampler(seed=42)
# )
# study_random.optimize(objective, n_trials=10)




[I 2026-11-24 11:13:55,784] A new study created in memory with name: no-name-acba711b-fd73-4426-a91d-2d7ea64a52c0
[I 2026-11-24 11:14:01,129] Trial 0 finished with value: 0.959633597267505 and parameters: {'n_estimators': 106, 'max_depth': 20, 'max_features': 0.7587945476302645, 'criterion': 'gini'}. Best is trial 0 with value: 0.959633597267505.
[I 2026-11-24 11:14:02,834] Trial 1 finished with value: 0.9648657040832169 and parameters: {'n_estimators': 58, 'max_depth': 18, 'max_features': 0.6410035105688879, 'criterion': 'log_loss'}. Best is trial 1 with value: 0.9648657040832169.
[I 2026-11-24 11:14:05,847] Trial 2 finished with value: 0.9648501785437045 and parameters: {'n_estimators': 175, 'max_depth': 6, 'max_features': 0.26364247048639056, 'criterion': 'log_loss'}. Best is trial 1 with value: 0.9648657040832169.
[I 2026-11-24 11:14:10,236] Trial 3 finished with value: 0.968390001552554 and parameters: {'n_estimators': 115, 'max_depth': 8, 'max_features': 0.6506676052501416, 'crit

RandomSampler Best Accuracy: 0.968390001552554
RandomSampler Best Params: {'n_estimators': 115, 'max_depth': 8, 'max_features': 0.6506676052501416, 'criterion': 'log_loss'}


In [44]:
# Uncomment all to run this

# print("RandomSampler Best Accuracy:", study_random.best_value)
# print("RandomSampler Best Params:", study_random.best_params)

RandomSampler Best Accuracy: 0.968390001552554
RandomSampler Best Params: {'n_estimators': 115, 'max_depth': 8, 'max_features': 0.6506676052501416, 'criterion': 'log_loss'}


### Using Grid Smapler in optuna

In [47]:
# Uncomment all to run this


# import optuna
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.model_selection import cross_val_score


# # Define the objective function
# def objective(trial):
#     n_estimators = trial.suggest_int("n_estimators", 50, 200)
#     max_depth = trial.suggest_int("max_depth", 3, 20)
#     max_features = trial.suggest_float("max_features", 0.1, 1.0)
#     criterion = trial.suggest_categorical(
#         "criterion", ["gini", "entropy", "log_loss"]
#     )

#     model = RandomForestClassifier(
#         n_estimators=n_estimators,
#         max_depth=max_depth,
#         max_features=max_features,
#         criterion=criterion,
#         random_state=42,
#     )

#     score = cross_val_score(model, X, y, scoring="accuracy", cv=5).mean()
#     return score


# # Define exact grid values for every parameter in objective()
# search_space = {
#     "n_estimators":[50,100,150],  # 3 choices
#     "max_depth":[5,6,7],  # 3 choices
#     "max_features": [0.2, 0.5, 0.8],  # 3 choices
#     "criterion": ["gini", "entropy", "log_loss"],  # 3 choices
# }
# # Total combinations = 3 * 3 * 3 * 3 = 81 trials

# # Create study with GridSampler
# study_grid = optuna.create_study(
#     direction="maximize", sampler=optuna.samplers.GridSampler(search_space)
# )
# study_grid.optimize(objective)  # Automatically runs all 81 grid combinations





[I 2026-11-24 11:16:47,425] A new study created in memory with name: no-name-c3067bdf-2dc5-46fa-90cf-14c556280b0e
[I 2026-11-24 11:16:50,355] Trial 0 finished with value: 0.9613724576929048 and parameters: {'n_estimators': 100, 'max_depth': 7, 'max_features': 0.5, 'criterion': 'gini'}. Best is trial 0 with value: 0.9613724576929048.
[I 2026-11-24 11:16:51,196] Trial 1 finished with value: 0.9613414066138798 and parameters: {'n_estimators': 50, 'max_depth': 5, 'max_features': 0.2, 'criterion': 'entropy'}. Best is trial 0 with value: 0.9613724576929048.
[I 2026-11-24 11:16:55,604] Trial 2 finished with value: 0.9666200900481291 and parameters: {'n_estimators': 100, 'max_depth': 5, 'max_features': 0.8, 'criterion': 'log_loss'}. Best is trial 2 with value: 0.9666200900481291.
[I 2026-11-24 11:16:58,112] Trial 3 finished with value: 0.9683900015525537 and parameters: {'n_estimators': 100, 'max_depth': 6, 'max_features': 0.5, 'criterion': 'gini'}. Best is trial 3 with value: 0.96839000155255

In [48]:
# Un comment all to run this

# print("GridSampler Best Accuracy:", study_grid.best_value)
# print("GridSampler Best Params:", study_grid.best_params)

GridSampler Best Accuracy: 0.9683900015525537
GridSampler Best Params: {'n_estimators': 100, 'max_depth': 6, 'max_features': 0.5, 'criterion': 'gini'}


# Optuna Visualizations

In [49]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [54]:
# 1. Optimization History
plot_optimization_history(study).show()

# important - show what we can use the trial number like trial = 33 is enough

In [55]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()

In [56]:
# 3. Slice Plot
plot_slice(study).show()

In [57]:
# 4. Contour Plot
plot_contour(study).show()

In [58]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

#  Here comes the most important point
# The optuna gives us a way of creatimg a dunamic search space
# Simply means it also can select the model which gives best  result
# Gives best models + their best hyperparaeters

In [59]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [60]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [61]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-11-24 11:43:04,618] A new study created in memory with name: no-name-0da1f475-9d5a-4864-a115-072f6d625764
[I 2026-11-24 11:43:04,659] Trial 0 finished with value: 0.9698108908635223 and parameters: {'classifier': 'SVM', 'C': 7.318447967577863, 'kernel': 'rbf', 'gamma': 'auto'}. Best is trial 0 with value: 0.9698108908635223.
[I 2026-11-24 11:43:04,691] Trial 1 finished with value: 0.9522480443533073 and parameters: {'classifier': 'SVM', 'C': 9.414613390928364, 'kernel': 'poly', 'gamma': 'scale'}. Best is trial 0 with value: 0.9698108908635223.
[I 2026-11-24 11:43:06,801] Trial 2 finished with value: 0.9396977291714134 and parameters: {'classifier': 'RandomForest', 'n_estimators': 286, 'max_depth': 16, 'min_samples_split': 8, 'min_samples_leaf': 6, 'bootstrap': True}. Best is trial 0 with value: 0.9698108908635223.
[I 2026-11-24 11:43:06,833] Trial 3 finished with value: 0.9472355130249867 and parameters: {'classifier': 'SVM', 'C': 0.3010301992938401, 'kernel': 'rbf', 'gamma': '

In [62]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)


# It tells us that svm is the best model
# and the best hyperparameter values of svm are given



Best trial parameters: {'classifier': 'SVM', 'C': 1.450656276107445, 'kernel': 'rbf', 'gamma': 'auto'}
Best trial accuracy: 0.9723171565276828


In [63]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,system_attrs_tpe:relative_params:0,state
0,0,0.969811,2026-11-24 11:43:04.625573,2026-11-24 11:43:04.659151,0 days 00:00:00.033578,7.318448,NaN,SVM,auto,rbf,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
1,1,0.952248,2026-11-24 11:43:04.660966,2026-11-24 11:43:04.691771,0 days 00:00:00.030805,9.414613,NaN,SVM,scale,poly,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
2,2,0.939698,2026-11-24 11:43:04.693933,2026-11-24 11:43:06.801876,0 days 00:00:02.107943,NaN,True,RandomForest,NaN,NaN,NaN,16.0,6.0,8.0,286.0,NaN,COMPLETE
3,3,0.947236,2026-11-24 11:43:06.803292,2026-11-24 11:43:06.833205,0 days 00:00:00.029913,0.301030,NaN,SVM,scale,rbf,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
4,4,0.937172,2026-11-24 11:43:06.834530,2026-11-24 11:43:08.424345,0 days 00:00:01.589815,NaN,False,RandomForest,NaN,NaN,NaN,10.0,9.0,4.0,243.0,NaN,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.969792,2026-11-24 11:43:56.356042,2026-11-24 11:43:56.399369,0 days 00:00:00.043327,1.986423,NaN,SVM,auto,rbf,NaN,NaN,NaN,NaN,NaN,"{""classifier"": ""SVM""}",COMPLETE
96,96,0.959748,2026-11-24 11:43:56.400593,2026-11-24 11:43:56.439126,0 days 00:00:00.038533,2.587933,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,"{""classifier"": ""SVM""}",COMPLETE
97,97,0.949761,2026-11-24 11:43:56.442279,2026-11-24 11:43:56.479599,0 days 00:00:00.037320,0.710421,NaN,SVM,auto,sigmoid,NaN,NaN,NaN,NaN,NaN,"{""classifier"": ""SVM""}",COMPLETE
98,98,0.972317,2026-11-24 11:43:56.482374,2026-11-24 11:43:56.522610,0 days 00:00:00.040236,1.659643,NaN,SVM,auto,rbf,NaN,NaN,NaN,NaN,NaN,"{""classifier"": ""SVM""}",COMPLETE


In [64]:
study.trials_dataframe()['params_classifier'].value_counts()

,count
params_classifier,
SVM,79
RandomForest,12
GradientBoosting,9


In [65]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

,value
params_classifier,
GradientBoosting,0.944987
RandomForest,0.942204
SVM,0.958871


# It is the most important , must try on all project